In [1]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
import numpy as np
import pandas as pd
from shapely.geometry import mapping

# -----------------------------
# Inputs
# -----------------------------
polygons_path = r"C:\Users\KyleSteen.AzureAD\Documents\NLCD\Python_Workspace\CONUS\CONUS_3_3.shp"
nlcd_path = r"C:\Users\KyleSteen.AzureAD\Documents\NLCD\Python_Workspace\CONUS\NLCD_CONUS_5070.tif"
out_csv = r"C:\Users\KyleSteen.AzureAD\Documents\NLCD\Python_Workspace\CONUS\NLCD_CONUS.csv"

# -----------------------------
# NLCD classes
# -----------------------------
nlcd_classes = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

nodata_val = 255

# -----------------------------
# Logger
# -----------------------------
def log(msg):
    print(f"[{pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)

# -----------------------------
# Main Processing
# -----------------------------
log("Loading shapefile")
gdf = gpd.read_file(polygons_path)

# Force ROW_ID to be the dataframe index
gdf = gdf.set_index("ROW_ID", drop=False)

# Safety check
assert gdf["ROW_ID"].is_unique, "ROW_ID must be unique"

if "Square_Met" not in gdf.columns:
    gdf["Square_Met"] = None

log("Fixing invalid geometries")
gdf["geometry"] = gdf["geometry"].buffer(0)

total_polys = len(gdf)
log(f"Total polygons: {total_polys}")

results = []

log("Opening raster")
with rasterio.open(nlcd_path) as src:

    for i, (row_id, row) in enumerate(gdf.iterrows(), start=1):

        geom = row.geometry
        square_met = row["Square_Met"]

        try:
            minx, miny, maxx, maxy = geom.bounds
            window = src.window(minx, miny, maxx, maxy)

            width = max(int(np.ceil(window.width)), 1)
            height = max(int(np.ceil(window.height)), 1)

            safe_window = Window(
                int(window.col_off),
                int(window.row_off),
                width,
                height
            )

            data = src.read(1, window=safe_window)

            mask = rasterize(
                [(mapping(geom), 1)],
                out_shape=data.shape,
                transform=src.window_transform(safe_window),
                fill=0,
                all_touched=True,
                dtype="uint8",
            )

            captured_pixels = np.count_nonzero(mask)

            if captured_pixels == 0:
                results.append((row_id, square_met, -1, "No Data", 0))
                continue

            values = data[mask == 1]
            values = values[values != nodata_val]

            if len(values) == 0:
                results.append((row_id, square_met, -1, "No Data", captured_pixels))
                continue

            counts = np.bincount(values.astype(np.int32))
            mode_val = np.argmax(counts)
            mode_class = nlcd_classes.get(mode_val, "Unknown")

            results.append((row_id, square_met, int(mode_val), mode_class, int(captured_pixels)))

        except Exception:
            results.append((row_id, square_met, -1, "Error", 0))

        # Progress update every 25k polygons
        if i % 25000 == 0 or i == total_polys:
            pct = (i / total_polys) * 100
            log(f"Progress: {pct:.1f}% ({i}/{total_polys})")

log("Writing CSV")

df_out = pd.DataFrame(
    results,
    columns=["ROW_ID", "Square_Met", "NLCD_Code", "NLCD_Class", "Pixels_Captured"]
)

df_out.to_csv(out_csv, index=False)

log("Done")

[2026-03-09 14:30:05] Loading shapefile
[2026-03-09 14:30:15] Fixing invalid geometries
[2026-03-09 14:30:31] Total polygons: 835996
[2026-03-09 14:30:31] Opening raster
[2026-03-09 14:30:51] Progress: 3.0% (25000/835996)
[2026-03-09 14:31:11] Progress: 6.0% (50000/835996)
[2026-03-09 14:31:30] Progress: 9.0% (75000/835996)
[2026-03-09 14:31:49] Progress: 12.0% (100000/835996)
[2026-03-09 14:32:09] Progress: 15.0% (125000/835996)
[2026-03-09 14:32:28] Progress: 17.9% (150000/835996)
[2026-03-09 14:32:48] Progress: 20.9% (175000/835996)
[2026-03-09 14:33:08] Progress: 23.9% (200000/835996)
[2026-03-09 14:33:27] Progress: 26.9% (225000/835996)
[2026-03-09 14:33:47] Progress: 29.9% (250000/835996)
[2026-03-09 14:34:08] Progress: 32.9% (275000/835996)
[2026-03-09 14:34:29] Progress: 35.9% (300000/835996)
[2026-03-09 14:34:49] Progress: 38.9% (325000/835996)
[2026-03-09 14:35:09] Progress: 41.9% (350000/835996)
[2026-03-09 14:35:28] Progress: 44.9% (375000/835996)
[2026-03-09 14:35:48] Prog